<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/6_2_Kernel_PCA_and_Kernel_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VI. Kernel Methods

## Kernel Trick, Mercer's Theorem, and Kernel Methods

## Kernel PCA and Kernel Logistic Regression

## Gaussian Processes

### 2. Ядровый PCA и ядровая логистическая регрессия

#### 1. PCA в пространстве признаков: центрирование и собственные векторы

Ядровой трюк, изученный нами в предыдущих разделах, не ограничивается задачами обучения с учителем. Его можно с равным успехом применить и к методам снижения размерности и выделения признаков без учителя. Классический пример — метод главных компонент (PCA). Перенесённый в спрямляющее пространство $\mathcal{H}$, он превращается в **ядровый PCA** (Kernel PCA) — мощный инструмент нелинейного анализа данных, позволяющий находить направления максимальной дисперсии в бесконечномерных признаковых пространствах, не вычисляя явно отображение $\phi$. В этом разделе мы выведем алгоритм Kernel PCA, начиная с обычного линейного PCA и последовательно заменяя все операции, зависящие от явных координат, их ядровыми аналогами.

---

##### 1.1. Линейный PCA как проекция на главные компоненты

Напомним кратко классический PCA. Пусть дана матрица центрированных данных $\mathbf{X} \in \mathbb{R}^{n \times D}$, строки которой — наблюдения $\mathbf{x}_i^\top \in \mathbb{R}^D$, причём $\sum_{i=1}^n \mathbf{x}_i = \mathbf{0}$. Выборочная ковариационная матрица равна $\mathbf{C} = \frac{1}{n} \mathbf{X}^\top \mathbf{X} \in \mathbb{R}^{D \times D}$. Главные компоненты — это ортонормированные собственные векторы $\mathbf{u}_k$ матрицы $\mathbf{C}$, соответствующие наибольшим собственным числам $\lambda_k$:

$$
\mathbf{C} \mathbf{u}_k = \lambda_k \mathbf{u}_k, \qquad \mathbf{u}_k^\top \mathbf{u}_l = \delta_{kl}.
$$

Проекция точки $\mathbf{x}_i$ на $k$-ю главную компоненту есть скаляр $\langle \mathbf{x}_i, \mathbf{u}_k \rangle$.

Альтернативно, можно рассмотреть матрицу Грама $\mathbf{G} = \mathbf{X} \mathbf{X}^\top \in \mathbb{R}^{n \times n}$, элементы которой суть попарные скалярные произведения $\mathbf{G}_{ij} = \mathbf{x}_i^\top \mathbf{x}_j$. Нетрудно показать, что если $\mathbf{v}_k$ — собственный вектор $\mathbf{G}$ с собственным числом $\mu_k$, то $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \mathbf{X}^\top \mathbf{v}_k$ является нормированным собственным вектором ковариационной матрицы $\mathbf{C}$, а $\mu_k = n \lambda_k$. Эта двойственность — ключ к ядровому обобщению: она позволяет выразить всё через скалярные произведения, избегая явной работы в $D$-мерном пространстве признаков.

---

##### 1.2. Центрирование в пространстве признаков

Перенесём эти идеи в RKHS. Пусть $\phi : \mathcal{X} \to \mathcal{H}$ — отображение в спрямляющее пространство, и $\Phi = [\phi(\mathbf{x}_1), \dots, \phi(\mathbf{x}_n)]$ — (неявная) матрица «признаков» размера $\dim(\mathcal{H}) \times n$. Для выполнения PCA в $\mathcal{H}$ данные должны быть центрированы — среднее отображённых векторов должно равняться нулю. Однако мы не можем вычислить среднее $\frac{1}{n} \sum_i \phi(\mathbf{x}_i)$ явно, если $\mathcal{H}$ бесконечномерно. Но можно работать непосредственно с центрированной ядерной матрицей.

Центрированные образы имеют вид $\tilde{\phi}(\mathbf{x}_i) = \phi(\mathbf{x}_i) - \frac{1}{n} \sum_{j=1}^n \phi(\mathbf{x}_j)$. В матричной форме:

$$
\tilde{\Phi} = \Phi - \frac{1}{n} \Phi \mathbf{1} \mathbf{1}^\top = \Phi \mathbf{H},
$$

где $\mathbf{H} = \mathbf{I} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top$ — центрирующая матрица ($\mathbf{1}$ — вектор из $n$ единиц). Матрица $\mathbf{H}$ симметрична, идемпотентна ($\mathbf{H}^2 = \mathbf{H}$) и обнуляет среднее: $\mathbf{H} \mathbf{1} = \mathbf{0}$.

Матрица Грама для центрированных образов, т.е. центрированная ядерная матрица $\tilde{\mathbf{K}}$, выражается через исходную $\mathbf{K}_{ij} = \langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle_{\mathcal{H}}$ следующим образом:

$$
\tilde{\mathbf{K}} = \tilde{\Phi}^\top \tilde{\Phi} = \mathbf{H} \Phi^\top \Phi \mathbf{H} = \mathbf{H} \mathbf{K} \mathbf{H}. \tag{1.1}
$$

Раскрывая, получаем явную формулу, не требующую знания $\phi$:

$$
\tilde{\mathbf{K}} = \mathbf{K} - \frac{1}{n} \mathbf{K} \mathbf{1} \mathbf{1}^\top - \frac{1}{n} \mathbf{1} \mathbf{1}^\top \mathbf{K} + \frac{1}{n^2} (\mathbf{1}^\top \mathbf{K} \mathbf{1}) \mathbf{1} \mathbf{1}^\top. \tag{1.2}
$$

Таким образом, центрирование в $\mathcal{H}$ сводится к чисто алгебраической операции над матрицей $\mathbf{K}$, вычислимой за $O(n^2)$.

---

##### 1.3. Алгоритм Kernel PCA

Имея центрированную ядерную матрицу $\tilde{\mathbf{K}}$, мы можем выполнить PCA в $\mathcal{H}$, опираясь исключительно на собственное разложение $\tilde{\mathbf{K}}$. Последовательность шагов такова:

1. **Вычислить матрицу ядра** $\mathbf{K}$ для обучающих данных: $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$.
2. **Центрировать:** $\tilde{\mathbf{K}} = \mathbf{H} \mathbf{K} \mathbf{H}$ по формуле (1.1) или (1.2).
3. **Найти собственные векторы** $\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n$ матрицы $\tilde{\mathbf{K}}$ и соответствующие собственные числа $\mu_1 \ge \mu_2 \ge \dots \ge \mu_n \ge 0$. Собственные векторы считаем ортонормированными в $\mathbb{R}^n$: $\mathbf{v}_k^\top \mathbf{v}_l = \delta_{kl}$.
4. **Связать с главными компонентами в $\mathcal{H}$.** Вектор главных компонент в $\mathcal{H}$ имеет вид $\mathbf{u}_k = \sum_{i=1}^n (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$. Чтобы $\mathbf{u}_k$ был ортонормирован в $\mathcal{H}$, коэффициент нормировки должен быть $1/\sqrt{\mu_k}$. Действительно,
   $$
   \|\mathbf{u}_k\|_{\mathcal{H}}^2 = \sum_{i,j} (\mathbf{v}_k)_i (\mathbf{v}_k)_j \tilde{K}_{ij} = \mathbf{v}_k^\top \tilde{\mathbf{K}} \mathbf{v}_k = \mu_k \|\mathbf{v}_k\|^2 = \mu_k,
   $$
   поэтому нормированный собственный вектор равен $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \sum_{i=1}^n (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$.

Таким образом, мы получаем ортонормированный базис главных направлений в $\mathcal{H}$, не покидая пространства коэффициентов.


##### 1.4. Проекция новой точки

Одно из важнейших применений PCA — понижение размерности новых наблюдений. Для ядрового PCA проекция тестовой точки $\mathbf{x}_*$ на $k$-ю главную компоненту даётся скалярным произведением

$$
z_k(\mathbf{x}_*) = \langle \tilde{\phi}(\mathbf{x}_*), \mathbf{u}_k \rangle_{\mathcal{H}} = \frac{1}{\sqrt{\mu_k}} \sum_{i=1}^n (\mathbf{v}_k)_i \langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}.
$$

Осталось выразить центрированное скалярное произведение $\langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}$ через исходное ядро. Используя определение центрирования, получаем

$$
\langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}} = K(\mathbf{x}_*, \mathbf{x}_i) - \frac{1}{n} \sum_{j=1}^n K(\mathbf{x}_*, \mathbf{x}_j) - \frac{1}{n} \sum_{j=1}^n K(\mathbf{x}_j, \mathbf{x}_i) + \frac{1}{n^2} \sum_{j,l} K(\mathbf{x}_j, \mathbf{x}_l).
$$

В векторной форме: определим столбец $\mathbf{k}_* = [K(\mathbf{x}_*, \mathbf{x}_1), \dots, K(\mathbf{x}_*, \mathbf{x}_n)]^\top$. Тогда центрированный вектор $\tilde{\mathbf{k}}_*$ получается как

$$
\tilde{\mathbf{k}}_* = \mathbf{k}_* - \frac{1}{n} \mathbf{K} \mathbf{1} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top \mathbf{k}_* + \frac{1}{n^2} (\mathbf{1}^\top \mathbf{K} \mathbf{1}) \mathbf{1}. \tag{1.3}
$$

Проекция на $k$-ю компоненту:

$$
z_k(\mathbf{x}_*) = \frac{1}{\sqrt{\mu_k}} \, \tilde{\mathbf{k}}_*^\top \mathbf{v}_k. \tag{1.4}
$$

Обратим внимание: поскольку ранг $\tilde{\mathbf{K}}$ не превышает $n$, число ненулевых собственных значений не больше $n$, а значит, мы можем извлечь не более $n$ главных компонент.



#### Визуализация: линейный PCA vs. Kernel PCA

Запустите ячейку ниже, чтобы сравнить линейный PCA и ядровый PCA (с RBF-ядром) на данных, представляющих две вложенные окружности. Линейный PCA не способен разделить классы — он лишь поворачивает систему координат. Kernel PCA, напротив, отображает данные в пространство, где классы становятся линейно разделимыми.

In [ ]:
# @title Сравнение линейного PCA и Kernel PCA (RBF)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA

# Генерируем данные: два концентрических круга
np.random.seed(0)
n = 300
r1 = np.random.randn(n) * 0.3 + 2
theta1 = np.random.rand(n) * 2 * np.pi
X1 = np.column_stack([r1 * np.cos(theta1), r1 * np.sin(theta1)])

r2 = np.random.randn(n) * 0.3 + 1
theta2 = np.random.rand(n) * 2 * np.pi
X2 = np.column_stack([r2 * np.cos(theta2), r2 * np.sin(theta2)])

X = np.vstack([X1, X2])
y = np.hstack([np.ones(n), -np.ones(n)])

# Линейный PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Kernel PCA с RBF-ядром
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=1.0)
X_kpca = kpca.fit_transform(X)

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_pca[:n, 0], X_pca[:n, 1], c='blue', alpha=0.5, label='Класс +1')
axes[0].scatter(X_pca[n:, 0], X_pca[n:, 1], c='red', alpha=0.5, label='Класс -1')
axes[0].set_title('Линейный PCA')
axes[0].legend()
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_kpca[:n, 0], X_kpca[:n, 1], c='blue', alpha=0.5, label='Класс +1')
axes[1].scatter(X_kpca[n:, 0], X_kpca[n:, 1], c='red', alpha=0.5, label='Класс -1')
axes[1].set_title('Kernel PCA (RBF)')
axes[1].legend()
axes[1].set_xlabel('kPC1')
axes[1].set_ylabel('kPC2')

plt.tight_layout()
plt.show()


##### 1.5. Углублённый взгляд: связь Kernel PCA с многомерным шкалированием (MDS)

> **Для читателя с математической подготовкой.** Kernel PCA тесно связан с классическим многомерным шкалированием (MDS). Напомним, что MDS по матрице квадратов попарных расстояний $D_{ij} = \|\mathbf{x}_i - \mathbf{x}_j\|^2$ строит координаты точек, сохраняющие эти расстояния. Если выполнить MDS с ядром $K_{\text{cent}}(\mathbf{x}_i, \mathbf{x}_j) = -\frac12 D_{ij}$ и применить двойное центрирование $\mathbf{H} \mathbf{K} \mathbf{H}$, то полученные координаты в точности совпадают с проекциями на главные компоненты Kernel PCA с ядром, индуцированным расстоянием.
>
> В частности, если взять линейное ядро $K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top \mathbf{y}$ и центрированные данные, то $\tilde{\mathbf{K}} = \mathbf{X} \mathbf{X}^\top$. Собственные векторы $\tilde{\mathbf{K}}$ с точностью до масштаба дадут проекции на классические главные компоненты. Таким образом, Kernel PCA включает классический PCA как частный случай.
>
> Более того, MDS можно рассматривать как Kernel PCA с ядром, неявно заданным через расстояние. Для произвольной симметричной матрицы различий можно построить ядро, если матрица двойного центрирования $-\frac12 \mathbf{H} \mathbf{D} \mathbf{H}$ окажется неотрицательно определённой. Эта глубокая связь объединяет PCA, MDS и спектральное вложение и позволяет переносить геометрическую интуицию из одного метода в другой.



#### Вопросы для самопроверки

**Для начинающих:**

1. Зачем нужно центрировать данные в пространстве признаков при выполнении Kernel PCA? Что произойдёт, если этого не сделать?
2. Выпишите формулу центрированной ядерной матрицы $\tilde{\mathbf{K}}$ через исходную матрицу $\mathbf{K}$ и объясните каждое слагаемое.
3. Что представляет собой матрица $\mathbf{H} = \mathbf{I} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top$? Какими свойствами она обладает?
4. Как вычислить проекцию новой точки $\mathbf{x}_*$ на $k$-ю главную компоненту в пространстве $\mathcal{H}$, не зная отображения $\phi$?
5. Сколько главных компонент можно извлечь с помощью Kernel PCA на $n$ точках? Почему?

**Для профессионалов:**

1. Выведите формулу центрирования $\tilde{\mathbf{K}} = \mathbf{H} \mathbf{K} \mathbf{H}$ и покажите, что она эквивалентна выражению (1.2). Объясните роль свойства идемпотентности $\mathbf{H}$.
2. Докажите, что собственные векторы матрицы $\tilde{\mathbf{K}}$ связаны с собственными векторами ковариационной матрицы в $\mathcal{H}$: если $\tilde{\mathbf{K}} \mathbf{v}_k = \mu_k \mathbf{v}_k$, то $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \sum_i (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$ — собственный вектор ковариационной матрицы с тем же собственным числом $\mu_k/n$.
3. Объясните, почему число ненулевых собственных значений $\tilde{\mathbf{K}}$ не превышает $n$. Как это связано с размерностью пространства $\mathcal{H}$?
4. Сравните Kernel PCA с методом главных кривых (principal curves). В чём принципиальное различие в определении нелинейных направлений максимальной вариации?

### 2. Реализация Kernel PCA на Python и визуализация

В предыдущем разделе мы вывели математические основы ядрового PCA: центрирование в пространстве признаков, собственное разложение центрированной ядерной матрицы и проекцию новых точек. Теперь мы превратим эти формулы в вычислительный инструмент, написав собственную реализацию Kernel PCA на Python и сравнив её с промышленной библиотекой scikit-learn. В качестве тестового полигона выступит классическое нелинейное многообразие — «швейцарский рулет» (Swiss roll), на котором мы наглядно увидим, как ядровый PCA «разворачивает» скрученные данные в плоскость.

Следующая ячейка содержит полную реализацию класса `KernelPCA` с методами `fit` и `transform`.

In [ ]:
# @title Реализация Kernel PCA с нуля
import numpy as np
from scipy.spatial.distance import cdist

class KernelPCA:
    def __init__(self, n_components=2, kernel='rbf', gamma=1.0, degree=3, coef0=0.0):
        self.n_components = n_components
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0

        self.X_train = None
        self.alphas_ = None      # собственные векторы (n_components x n)
        self.lambdas_ = None     # собственные числа
        self.K_train = None      # матрица Грама обучающей выборки
        self.ones_col = None     # столбец единиц для центрирования

    def _compute_kernel(self, X, Y=None):
        if Y is None:
            Y = X
        if self.kernel == 'linear':
            K = X @ Y.T
        elif self.kernel == 'poly':
            K = (X @ Y.T * self.gamma + self.coef0) ** self.degree
        elif self.kernel == 'rbf':
            sq_dists = cdist(X, Y, 'sqeuclidean')
            K = np.exp(-self.gamma * sq_dists)
        elif self.kernel == 'sigmoid':
            K = np.tanh(self.gamma * (X @ Y.T) + self.coef0)
        else:
            raise ValueError(f'Unknown kernel: {self.kernel}')
        return K

    def fit(self, X):
        n = X.shape[0]
        self.X_train = X
        self.K_train = self._compute_kernel(X)

        # Центрирование
        H = np.eye(n) - 1.0 / n * np.ones((n, n))
        K_centered = H @ self.K_train @ H

        # Собственное разложение
        eigvals, eigvecs = np.linalg.eigh(K_centered)
        # Сортировка по убыванию
        idx = np.argsort(eigvals)[::-1]
        eigvals = eigvals[idx]
        eigvecs = eigvecs[:, idx]

        # Отбираем первые n_components, отсекаем отрицательные/малые
        mask = eigvals > 1e-12
        eigvals = eigvals[mask]
        eigvecs = eigvecs[:, mask]

        n_comp = min(self.n_components, len(eigvals))
        self.lambdas_ = eigvals[:n_comp]
        # Собственные векторы – столбцы; для удобства храним как (n_components, n)
        self.alphas_ = eigvecs[:, :n_comp].T

        # Вспомогательные величины для центрирования тестовых данных
        self.ones_col = np.ones((n, 1))
        self.K_train_row_mean = np.mean(self.K_train, axis=1, keepdims=True)
        self.K_train_mean = np.mean(self.K_train)

        return self

    def transform(self, X):
        if self.alphas_ is None:
            raise RuntimeError("Модель не обучена. Вызовите fit().")
        n_train = self.X_train.shape[0]
        n_new = X.shape[0]

        K_test = self._compute_kernel(X, self.X_train)  # (n_new, n_train)

        # Центрирование тестовой матрицы по формуле (1.3)
        K_test_centered = (K_test
                           - np.mean(K_test, axis=1, keepdims=True)
                           - self.K_train_row_mean.T
                           + self.K_train_mean)

        # Проекция: Z = K_test_centered @ alphas^T / sqrt(lambdas)
        alphas_T = self.alphas_.T  # (n_train, n_components)
        Z = K_test_centered @ alphas_T / np.sqrt(self.lambdas_)
        return Z

print("Класс KernelPCA загружен.")

#### Визуализация на «швейцарском рулете»

Сгенерируем трёхмерное множество точек, лежащих на свёрнутой поверхности, — Swiss roll. Каждая точка получает цвет, соответствующий углу $t$, что создаёт гладкий цветовой градиент. Мы применим линейный PCA и Kernel PCA с RBF‑ядром, чтобы продемонстрировать, как ядровый метод «разворачивает» многообразие.

Запустите ячейку ниже.

In [ ]:
# @title Swiss roll: линейный PCA vs Kernel PCA (исправлено)
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA as SklearnKernelPCA
from mpl_toolkits.mplot3d import Axes3D

# Генерация Swiss roll
np.random.seed(42)
n = 800
t = 3 * np.pi / 2 + np.random.rand(n) * 3 * np.pi
height = np.random.rand(n) * 4 - 2
X_swiss = np.column_stack([t * np.cos(t), height, t * np.sin(t)])
color = t

# Линейный PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_swiss)

# Наш Kernel PCA с RBF
kpca_our = KernelPCA(n_components=2, kernel='rbf', gamma=0.05)
kpca_our.fit(X_swiss)
X_kpca_our = kpca_our.transform(X_swiss)

# sklearn Kernel PCA (для сравнения)
kpca_sk = SklearnKernelPCA(n_components=2, kernel='rbf', gamma=0.05, fit_inverse_transform=False)
X_kpca_sk = kpca_sk.fit_transform(X_swiss)

# Визуализация
fig = plt.figure(figsize=(18, 6))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], c=color, cmap='viridis', s=10)
ax1.set_title('Исходный Swiss roll (3D)')

ax2 = fig.add_subplot(1, 3, 2)
ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=color, cmap='viridis', s=10)
ax2.set_title('Линейный PCA')

ax3 = fig.add_subplot(1, 3, 3)
ax3.scatter(X_kpca_our[:, 0], X_kpca_our[:, 1], c=color, cmap='viridis', s=10)
ax3.set_title('Kernel PCA (RBF)')

plt.tight_layout()
plt.show()

# Проверка совпадения со sklearn
diff = np.abs(np.corrcoef(X_kpca_our[:, 0], X_kpca_sk[:, 0])[0, 1])
print(f"Корреляция первой компоненты (our vs sklearn): {diff:.4f}")

#### Влияние параметра $\gamma$ RBF-ядра

Параметр $\gamma$ определяет характерный масштаб близости. При малых $\gamma$ ядро становится глобальным, при больших — локальным, вплоть до почти диагональной матрицы. Интерактивный слайдер ниже позволяет исследовать, как меняются проекции Kernel PCA при изменении $\gamma$.

In [ ]:
# @title Влияние γ на Kernel PCA (интерактивно, исправлено)
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider

def plot_kpca_gamma(gamma=0.05):
    kpca = KernelPCA(n_components=2, kernel='rbf', gamma=gamma)
    kpca.fit(X_swiss)
    X_proj = kpca.transform(X_swiss)

    plt.figure(figsize=(7, 6))
    plt.scatter(X_proj[:, 0], X_proj[:, 1], c=color, cmap='viridis', s=10)
    plt.title(f'Kernel PCA (RBF, $\\gamma$={gamma:.3f})')
    plt.xlabel('kPC1')
    plt.ylabel('kPC2')
    plt.grid(True, alpha=0.3)
    plt.show()

interact(plot_kpca_gamma,
         gamma=FloatLogSlider(value=0.05, base=10, min=-3, max=1, step=0.05,
                              description='γ'));

#### Вычислительная сложность и проблема масштабирования

Полная реализация Kernel PCA требует $O(n^2)$ памяти и $O(n^3)$ времени. При $n \approx 10^4$ это ещё приемлемо, но для $10^5$ и более необходимо применять приближённые методы: аппроксимацию Найстрёма, Landmark Kernel PCA или случайные признаки (Random Fourier Features). В учебной реализации можно предусмотреть опциональный параметр `n_landmarks`, при указании которого используется Найстрём-приближение.



#### Вопросы для самопроверки

**Для начинающих:**

1. Как центрировать тестовую точку при проекции в Kernel PCA? Почему нельзя просто вычислить $K(\mathbf{x}_*, \mathbf{x}_i)$ и подставить в формулу без центрирования?
2. Почему собственные векторы центрированной ядерной матрицы задают коэффициенты разложения, а не сами главные компоненты? Как получить направления в $\mathcal{H}$?
3. Как выбрать число главных компонент в Kernel PCA? Какие критерии можно использовать?
4. Зачем нужна нормировка собственных векторов на $1/\sqrt{\lambda_k}$? Что произойдёт, если её опустить?

**Для профессионалов:**

1. Реализуйте формулу центрирования тестовой ядерной матрицы без вычисления полной матрицы $\mathbf{K}_{\text{test}} \mathbf{1}\mathbf{1}^\top$ (используйте суммирование по столбцам). Сравните по времени с наивной версией.
2. Объясните, как интерпретировать первые две компоненты Kernel PCA на Swiss roll как нелинейные координаты, параметризующие многообразие. Как эта параметризация соотносится с истинными порождающими параметрами?
3. Сравните время работы вашей реализации и `sklearn.decomposition.KernelPCA` при разных $n$. Какие оптимизации использует библиотечная версия?
4. Добавьте в свой класс поддержку предвычисленного ядра: если на вход `fit` подаётся готовая матрица $\mathbf{K}$, пропускать шаг её вычисления. В каких сценариях это полезно?

### 3. Денузинг и реконструкция в Kernel PCA

Ядровый PCA не только вскрывает нелинейную структуру данных, но и может быть использован для их очистки от шума — **денузинга**. В линейном PCA восстановление очищенной точки по её проекции на главные компоненты тривиально: это линейная комбинация базисных векторов. В RKHS прямое восстановление $\hat{\phi}$ также является линейной операцией, однако ему соответствует точка в исходном пространстве $\mathcal{X}$ только тогда, когда образ $\hat{\phi}$ лежит в образе отображения $\phi(\mathcal{X})$. Поиск такой точки, называемый **задачей предобраза** (pre-image problem), составляет центральную трудность и, одновременно, изюминку нелинейного денузинга с помощью Kernel PCA.

---

#### 3.1. Идея денузинга: от линейного PCA к нелинейному

В линейном PCA данные $\mathbf{x} \in \mathbb{R}^D$ проецируются на $k$-мерное подпространство, натянутое на первые $k$ главных компонент $\mathbf{u}_1, \dots, \mathbf{u}_k$. Очищенная (денуизированная) версия точки $\mathbf{x}$ есть её проекция на это подпространство:

$$
\hat{\mathbf{x}} = \sum_{j=1}^k \langle \mathbf{x}, \mathbf{u}_j \rangle \mathbf{u}_j = \mathbf{U}_k \mathbf{U}_k^\top \mathbf{x},
$$

где столбцы матрицы $\mathbf{U}_k$ — векторы $\mathbf{u}_j$. Эта операция уменьшает шум, отбрасывая компоненты, соответствующие малым собственным числам.

В ядровом PCA мы работаем в спрямляющем пространстве $\mathcal{H}$. Для точки $\mathbf{x}$ её образ $\phi(\mathbf{x})$ проецируется на подпространство $\mathcal{H}_k$, натянутое на первые $k$ главных направлений $\mathbf{u}_1, \dots, \mathbf{u}_k \in \mathcal{H}$. Проекция в $\mathcal{H}$ имеет вид

$$
\hat{\phi}(\mathbf{x}) = \sum_{j=1}^k \langle \phi(\mathbf{x}), \mathbf{u}_j \rangle_{\mathcal{H}} \mathbf{u}_j.
$$

Используя представление $\mathbf{u}_j = \sum_{i=1}^n (\mathbf{v}_j)_i \tilde{\phi}(\mathbf{x}_i)$, где $\mathbf{v}_j$ — $j$-й собственный вектор центрированной ядерной матрицы $\tilde{\mathbf{K}}$ с собственным числом $\mu_j$, а $\tilde{\phi}(\mathbf{x}_i)$ — центрированные образы, мы можем записать $\hat{\phi}(\mathbf{x})$ как линейную комбинацию центрированных образов обучающих точек:

$$
\hat{\phi}(\mathbf{x}) = \sum_{i=1}^n \alpha_i \tilde{\phi}(\mathbf{x}_i), \qquad \alpha_i = \sum_{j=1}^k \frac{(\mathbf{v}_j)_i}{\sqrt{\mu_j}} \langle \tilde{\phi}(\mathbf{x}), \mathbf{u}_j \rangle_{\mathcal{H}} = \sum_{j=1}^k \frac{(\mathbf{v}_j)_i}{\mu_j} (\tilde{\mathbf{k}}_*^\top \mathbf{v}_j),
$$

где $\tilde{\mathbf{k}}_*$ — центрированный вектор ядер тестовой точки. Таким образом, проекция $\hat{\phi}$ в $\mathcal{H}$ вычисляется без знания $\phi$. Однако $\hat{\phi}$ — это элемент RKHS, и чтобы получить точку в исходном пространстве $\mathcal{X}$, нужно найти $\mathbf{z} \in \mathcal{X}$ такой, что $\phi(\mathbf{z}) \approx \hat{\phi}$. Это и есть **задача предобраза**.

---

#### 3.2. Поиск предобраза для RBF-ядра

Рассмотрим часто используемое RBF-ядро $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$. Задача поиска предобраза ставится как минимизация расстояния в $\mathcal{H}$ между образом искомой точки $\phi(\mathbf{z})$ и заданной линейной комбинацией $\boldsymbol{\psi} = \sum_{i=1}^n \alpha_i \tilde{\phi}(\mathbf{x}_i)$:

$$
\mathbf{z}^* = \arg\min_{\mathbf{z} \in \mathcal{X}} \|\phi(\mathbf{z}) - \boldsymbol{\psi}\|_{\mathcal{H}}^2.
$$

Раскроем квадрат нормы, используя тождество $\|\phi(\mathbf{z}) - \boldsymbol{\psi}\|^2 = \|\phi(\mathbf{z})\|^2 - 2\langle \phi(\mathbf{z}), \boldsymbol{\psi} \rangle + \|\boldsymbol{\psi}\|^2$. Для RBF-ядра $\|\phi(\mathbf{z})\|^2 = K(\mathbf{z}, \mathbf{z}) = 1$ постоянно, поэтому задача эквивалентна максимизации скалярного произведения

$$
J(\mathbf{z}) = \langle \phi(\mathbf{z}), \boldsymbol{\psi} \rangle_{\mathcal{H}} = \sum_{i=1}^n \alpha_i \langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}.
$$

Центрированное скалярное произведение $\langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}$ выражается через ядро аналогично (1.3):

$$
\langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}} = K(\mathbf{z}, \mathbf{x}_i) - \frac{1}{n}\sum_{j=1}^n K(\mathbf{z}, \mathbf{x}_j) - \frac{1}{n}\sum_{j=1}^n K(\mathbf{x}_j, \mathbf{x}_i) + \frac{1}{n^2}\sum_{j,l} K(\mathbf{x}_j, \mathbf{x}_l).
$$

Таким образом, $J(\mathbf{z})$ — гладкая функция от $\mathbf{z}$, и для её максимизации можно применить градиентный подъём. Градиент $J(\mathbf{z})$ для RBF-ядра вычисляется аналитически:

$$
\nabla_{\mathbf{z}} J(\mathbf{z}) = 2\gamma \sum_{i=1}^n \alpha_i \bigl[ - K(\mathbf{z}, \mathbf{x}_i) (\mathbf{z} - \mathbf{x}_i) + \frac{1}{n} \sum_{j=1}^n K(\mathbf{z}, \mathbf{x}_j) (\mathbf{z} - \mathbf{x}_j) \bigr].
$$

Запуская итерации $\mathbf{z}^{(t+1)} = \mathbf{z}^{(t)} + \eta \nabla_{\mathbf{z}} J(\mathbf{z}^{(t)})$ из подходящего начального приближения (например, ближайшей обучающей точки или её образа), мы находим предобраз $\mathbf{z}^*$, который и будет очищенной версией исходной точки.

Важно: предобраз определён неоднозначно. Например, точки, симметричные относительно многообразия, могут давать одинаковые проекции. Выбор начального приближения регулирует, в какой «бассейн притяжения» сойдётся оптимизация.

---

#### 3.3. Денузинг на примере двумерных точек

Проиллюстрируем процесс. Сгенерируем точки на одномерном нелинейном многообразии в $\mathbb{R}^2$ — например, на полуокружности или спирали, — и добавим к координатам гауссовский шум. Применим Kernel PCA с RBF-ядром к зашумлённым данным.

- Поскольку истинная размерность данных равна 1, сохраним одну главную компоненту ($k=1$), полагая, что она описывает форму многообразия, а отброшенные компоненты поглощают шум.
- Для каждой зашумлённой точки $\mathbf{x}_i$ вычислим её проекцию в $\mathcal{H}$ на первую главную компоненту: получим коэффициенты $\alpha_i$ и вектор $\boldsymbol{\psi}$.
- Решив задачу предобраза (градиентным подъёмом), получим точку $\mathbf{z}_i^*$ — восстановленную, очищенную от шума версию $\mathbf{x}_i$.

Визуализация показывает: шумные точки, разбросанные вокруг истинной кривой, после денузинга «подтягиваются» к ней, образуя гладкую линию. Это демонстрирует способность Kernel PCA выделять нелинейное многообразие, даже когда оно не задано явно.

---

#### 3.4. Сравнение с линейным PCA

Линейный PCA, применённый к тем же зашумлённым данным на полуокружности, спроецирует их на прямую — направление максимальной дисперсии. Эта прямая не способна отразить кривизну данных; денузинг с одной главной компонентой восстановит точки на прямой, что даст значительно худшее приближение к истинной полуокружности. Kernel PCA выигрывает именно за счёт нелинейного вложения: его главные направления в $\mathcal{H}$ соответствуют нелинейным кривым в $\mathcal{X}$, позволяя моделировать искривлённые многообразия.

Однако это преимущество не бесплатно: денузинг через предобраз требует решения нетривиальной оптимизационной задачи для каждой точки и может быть чувствителен к инициализации и выбору $\gamma$.



#### Визуализация денузинга: зашумлённая полуокружность

Проиллюстрируем процесс на конкретном примере. Сгенерируем точки на полуокружности, добавим гауссовский шум, применим Kernel PCA с $k=1$ компонентой и для каждой точки найдём предобраз с помощью градиентного подъёма. Восстановленные точки должны лечь на гладкую кривую, близкую к истинной полуокружности.

In [ ]:
# @title Денузинг с Kernel PCA (полуокружность)
import numpy as np
import matplotlib.pyplot as plt

# 1. Генерация данных: полуокружность + шум
np.random.seed(42)
n = 100
theta = np.random.rand(n) * np.pi
radius = 1.0
X_true = np.column_stack([radius * np.cos(theta), radius * np.sin(theta)])
X_noisy = X_true + np.random.normal(0, 0.15, size=(n, 2))

# 2. Kernel PCA с 1 компонентой
gamma = 2.0
kpca = KernelPCA(n_components=1, kernel='rbf', gamma=gamma)
kpca.fit(X_noisy)
X_proj = kpca.transform(X_noisy)  # (n, 1)

# 3. Поиск предобраза для каждой точки
def preimage_objective(z, alphas, X_train, gamma):
    """Максимизируем J(z) = <phi(z), psi>."""
    K_z = np.exp(-gamma * np.sum((z - X_train)**2, axis=1))  # (n_train,)
    return np.dot(alphas, K_z)

def preimage_gradient(z, alphas, X_train, gamma):
    K_z = np.exp(-gamma * np.sum((z - X_train)**2, axis=1))[:, np.newaxis]
    diff = z - X_train
    grad = -2 * gamma * np.sum(alphas[:, np.newaxis] * K_z * diff, axis=0)
    # Учитываем центрирование: для простоты используем нецентрированные векторы,
    # поэтому градиент приближённый. Можно добавить поправку, но эффект мал.
    return grad

X_denoised = np.zeros_like(X_noisy)
for i in range(n):
    alphas_i = np.sum(kpca.alphas_ * X_proj[i], axis=0) / np.sqrt(kpca.lambdas_)
    z = X_noisy[i].copy()
    for step in range(200):
        grad = preimage_gradient(z, alphas_i, kpca.X_train, gamma)
        z += 0.05 * grad
    X_denoised[i] = z

# 4. Визуализация
plt.figure(figsize=(8, 6))
plt.scatter(X_noisy[:, 0], X_noisy[:, 1], c='gray', alpha=0.5, label='Зашумлённые данные')
plt.scatter(X_denoised[:, 0], X_denoised[:, 1], c='red', s=30, label='После денузинга')
plt.plot(X_true[:, 0], X_true[:, 1], 'b-', linewidth=2, label='Истинная кривая')
plt.legend()
plt.title('Денузинг с Kernel PCA (RBF-ядро, k=1)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()



#### 3.5. Ограничения и особые случаи

- **Неединственность предобраза.** Как упоминалось, функция $J(\mathbf{z})$ может иметь множество локальных максимумов, особенно при больших $k$ или сильно зашумлённых данных. На практике используют несколько случайных стартов и выбирают наилучший результат.
- **Полиномиальное ядро: аналитическое решение.** Для полиномиального ядра $K(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^\top \mathbf{x}' + r)^d$ предобраз иногда можно найти в замкнутой форме. Например, для однородного квадратичного ядра $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}')^2$ задача сводится к поиску $\mathbf{z}$, минимизирующего невязку между матрицами $\mathbf{z}\mathbf{z}^\top$ и целевой матрицей, что решается через собственное разложение. В общем случае полиномиальные ядра позволяют выразить $\phi$ явно и тем самым свести задачу к линейной или квадратичной оптимизации.
- **Вычислительная сложность.** Каждая итерация градиентного подъёма требует $O(n)$ вычислений ядра, что для больших $n$ может быть затратно. Приближённые методы (Nyström, случайные признаки) помогают ускорить как прямое построение проекции, так и поиск предобраза.



#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое задача предобраза и почему она возникает при денузинге с Kernel PCA?
2. Почему в RBF-ядре норма $\|\phi(\mathbf{z})\|_{\mathcal{H}}$ постоянна и как это упрощает поиск предобраза?
3. Опишите, как градиентный подъём используется для нахождения предобраза в RBF-ядре.
4. Приведите пример данных, где Kernel PCA денузинг работает значительно лучше линейного PCA. Объясните, почему.
5. Чем отличается операция проекции в $\mathcal{H}$ от восстановления точки в исходном пространстве?

**Для профессионалов:**

1. Выведите выражение для градиента $J(\mathbf{z})$ в случае RBF-ядра и покажите, что оно представимо как взвешенная сумма разностей $\mathbf{z} - \mathbf{x}_i$.
2. Предложите альтернативный метод поиска предобраза, основанный на многомерном шкалировании (MDS). В чём его преимущества и недостатки перед градиентным спуском?
3. Сравните качество денузинга с помощью Kernel PCA для RBF-ядра и полиномиального ядра степени 2 на данных, лежащих на эллипсе. Какое ядро предпочтительнее и почему?
4. Объясните, почему для полиномиального ядра $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}')^d$ предобраз иногда может быть найден аналитически. Какие алгебраические свойства используются?

### 4. Ядровая логистическая регрессия: вывод и регуляризация в RKHS

До сих пор мы применяли ядровой трюк к задачам регрессии (KRR) и обучения без учителя (Kernel PCA). В задачах классификации естественным кандидатом на «ядровизацию» является логистическая регрессия — линейный метод, минимизирующий логарифмическую функцию потерь. Перенесённая в воспроизводящее ядерное гильбертово пространство (RKHS), она превращается в **ядерную логистическую регрессию** (Kernel Logistic Regression, KLR) — нелинейный классификатор, который строит гладкие, вероятностно интерпретируемые решающие границы. В этом разделе мы выведем KLR, исследуем свойства получаемого решения, обсудим метод его обучения (IRLS) и сравним с другим популярным ядровым классификатором — SVM.

---

#### 4.1. Постановка задачи и применение теоремы о представителе

Пусть дана обучающая выборка $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, $\mathbf{x}_i \in \mathcal{X}$, $y_i \in \{-1, +1\}$. Модель ядерной логистической регрессии задаёт условную вероятность класса через сигмоидную функцию от элемента $f \in \mathcal{H}$, где $\mathcal{H}$ — RKHS с воспроизводящим ядром $K$:

$$
p(y = +1 \mid \mathbf{x}) = \sigma(f(\mathbf{x})), \qquad \sigma(t) = \frac{1}{1 + e^{-t}}.
$$

При $y \in \{-1, +1\}$ логарифмическая функция потерь (кросс-энтропия) для одного примера записывается как $\log(1 + \exp(-y f(\mathbf{x})))$. Для обучения выбираем регуляризованный эмпирический риск:

$$
\min_{f \in \mathcal{H}} \left\{ J(f) = \sum_{i=1}^n \log\!\bigl(1 + \exp(-y_i f(\mathbf{x}_i))\bigr) + \frac{\lambda}{2} \|f\|_{\mathcal{H}}^2 \right\}, \qquad \lambda > 0. \tag{4.1}
$$

Слагаемое $\frac{\lambda}{2}\|f\|_{\mathcal{H}}^2$ играет роль штрафа за сложность функции, контролируя гладкость и предотвращая переобучение.

Согласно теореме о представителе (раздел 3.5), решение задачи (4.1) существует и имеет вид конечной линейной комбинации ядер, центрированных в обучающих точках:

$$
f^*(\mathbf{x}) = \sum_{i=1}^n \alpha_i K(\mathbf{x}, \mathbf{x}_i), \qquad \boldsymbol{\alpha} \in \mathbb{R}^n. \tag{4.2}
$$

Подставляя (4.2) в (4.1) и используя матрицу Грама $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$, получаем конечномерную выпуклую задачу оптимизации относительно вектора коэффициентов $\boldsymbol{\alpha}$:

$$
\min_{\boldsymbol{\alpha} \in \mathbb{R}^n} \left\{ \hat{J}(\boldsymbol{\alpha}) = \sum_{i=1}^n \log\!\bigl(1 + \exp(-y_i (\mathbf{K}\boldsymbol{\alpha})_i)\bigr) + \frac{\lambda}{2} \boldsymbol{\alpha}^\top \mathbf{K} \boldsymbol{\alpha} \right\}. \tag{4.3}
$$

В отличие от задачи KRR, здесь функция потерь не квадратична, поэтому аналитическое решение отсутствует. Однако выпуклость и гладкость позволяют эффективно применять методы ньютоновского типа.

---

#### 4.2. Свойства решения и связь с SVM

Ядерная логистическая регрессия обладает несколькими важными свойствами, которые определяют её место среди ядровых классификаторов.

1.  **Вероятностный выход.** По построению модель оценивает вероятность $p(y = +1 \mid \mathbf{x})$, что полезно во многих приложениях (ранжирование, оценка рисков, калибровка).

2.  **Гладкость и дифференцируемость.** Логарифмическая функция потерь является гладкой и строго выпуклой по $f$, что позволяет использовать методы второго порядка (Ньютон, IRLS) и гарантирует быструю сходимость.

3.  **Отсутствие разреженности.** В отличие от SVM с hinge loss, где многие коэффициенты $\alpha_i$ становятся нулевыми (опорные векторы), в KLR, как правило, все $\alpha_i \neq 0$. Это связано с тем, что логистическая потеря не имеет «плоского» участка, и каждое наблюдение вносит вклад в решение. В результате предсказание требует вычисления ядра со всеми обучающими точками, что делает KLR более затратным на этапе тестирования.

4.  **Связь с максимумом правдоподобия.** При $\lambda \to 0$ KLR стремится к нерегуляризованному максимуму правдоподобия в RKHS. В случае линейного ядра получается классическая логистическая регрессия с $\ell_2$-штрафом (гребневая логистическая регрессия).

5.  **Сравнение с SVM.** SVM минимизирует hinge loss $\max(0, 1 - y f(\mathbf{x}))$, что даёт разреженное решение и нацелено на максимизацию зазора. Hinge loss менее чувствителен к выбросам, но не даёт вероятностей напрямую — для калибровки требуется постобработка (Platt scaling). KLR, напротив, естественно калибрована, но может хуже разделять классы при малых выборках из-за отсутствия встроенного механизма зазора. На практике выбор между ними диктуется требованиями к вероятностной интерпретации и вычислительными ограничениями.

---

#### 4.3. Оптимизация: метод Ньютона и IRLS в RKHS

Для минимизации (4.3) удобно использовать метод Ньютона, который в контексте обобщённых линейных моделей принимает форму итеративно перевзвешиваемого метода наименьших квадратов (Iteratively Reweighted Least Squares, IRLS). Выпишем градиент и гессиан функции $\hat{J}(\boldsymbol{\alpha})$.

Введём обозначение $\mathbf{f} = \mathbf{K}\boldsymbol{\alpha}$ — вектор значений функции на обучающих точках. Тогда логарифмическая потеря для $i$-го примера есть $L_i = \log(1 + e^{-y_i f_i})$. Производная $L_i$ по $f_i$ равна $-\frac{y_i}{1 + e^{y_i f_i}} = -y_i (1 - \sigma(y_i f_i))$. Определим вектор «откликов» и веса:

$$
p_i = \sigma(y_i f_i) = \frac{1}{1 + e^{-y_i f_i}}, \qquad
w_i = p_i (1 - p_i) = \frac{e^{-y_i f_i}}{(1 + e^{-y_i f_i})^2}.
$$

Тогда градиент $\hat{J}$ по $\boldsymbol{\alpha}$:

$$
\nabla_{\boldsymbol{\alpha}} \hat{J} = \mathbf{K} \mathbf{r} + \lambda \mathbf{K} \boldsymbol{\alpha}, \quad \text{где} \quad \mathbf{r}_i = -y_i (1 - p_i).
$$

Гессиан $\hat{J}$:

$$
\nabla^2_{\boldsymbol{\alpha}} \hat{J} = \mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K},
$$

где $\mathbf{W} = \operatorname{diag}(w_1, \dots, w_n)$ — диагональная матрица весов, $w_i > 0$. Гессиан положительно определён при $\lambda > 0$ и положительно полуопределённой $\mathbf{K}$, что гарантирует выпуклость.

Шаг метода Ньютона: $\boldsymbol{\alpha}_{\text{new}} = \boldsymbol{\alpha}_{\text{old}} - (\nabla^2 \hat{J})^{-1} \nabla \hat{J}$. Алгебраические преобразования позволяют переписать этот шаг в эквивалентной форме как решение взвешенной линейной системы относительно нового $\boldsymbol{\alpha}$:

$$
(\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{K} \mathbf{W} \mathbf{z},
$$

где $\mathbf{z} = \mathbf{f} + \mathbf{W}^{-1} (\mathbf{y} - \mathbf{p})$ — скорректированный отклик. Эту систему можно эффективно решать, например, через двойственное представление или с использованием разложения Холецкого, если $n$ не слишком велико. Итерации продолжаются до сходимости.

Вычислительная сложность одной итерации — $O(n^3)$ при прямом обращении; на практике применяют итерационные решатели или метод сопряжённых градиентов, особенно для больших $n$. Также возможен вывод в прямой форме (с явным $\phi$) через формулу Вудбери, аналогично KRR.



#### 4.4. Углублённый взгляд: калибровка вероятностей в KLR против SVM

> **Для читателя с математической подготовкой.** Одним из ключевых преимуществ KLR является теоретическая гарантия калибровки вероятностей. Минимизация логарифмической функции потерь эквивалентна максимизации правдоподобия Бернулли, поэтому при достаточно богатом RKHS и правильной спецификации модели предсказанные вероятности $\hat{p}(y=1 \mid \mathbf{x}) = \sigma(f^*(\mathbf{x}))$ асимптотически сходятся к истинным апостериорным вероятностям.
>
> SVM, напротив, нацелен на оценку разделяющей поверхности, а не вероятностей. Его функция потерь (hinge) не является скор-функцией (strictly proper scoring rule), и выход $f(\mathbf{x})$ не имеет вероятностной шкалы. На практике применяют метод Platt scaling: подгоняют дополнительную сигмоиду $\hat{p} = \sigma(A f(\mathbf{x}) + B)$ на отложенной выборке. Однако такая калибровка может быть ненадёжной в областях с малой плотностью данных или при мультимодальных распределениях.
>
> KLR не требует постобработки и даёт хорошо откалиброванные вероятности во всей области, что особенно ценно в задачах, где важна не только точность классификации, но и качество вероятностных прогнозов (например, в медицине, финансах, рекомендательных системах). Платой служит отсутствие встроенного механизма разреженности и, как следствие, большие вычислительные затраты на этапе предсказания.


#### Сравнение KLR и SVM на нелинейных данных

Чтобы увидеть практическую разницу между ядерной логистической регрессией и SVM, построим их решающие границы на данных типа «луны» (два пересекающихся полумесяца). Обратите внимание: KLR даёт гладкую вероятностную границу с оценкой принадлежности классу, тогда как SVM ориентируется на отступ и выдаёт разреженное решение (опорные векторы отмечены кружками).

In [ ]:
# @title Сравнение KLR и SVM на нелинейных данных
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.svm import SVC
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Генерация данных
X, y = make_moons(n_samples=200, noise=0.3, random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# SVM с RBF-ядром
svm = SVC(kernel='rbf', gamma=1.0, C=1.0, probability=True)
svm.fit(X_scaled, y)

# Ядерная логистическая регрессия (приближение через случайные признаки)
rbf_feature = RBFSampler(gamma=1.0, n_components=200, random_state=42)
X_rbf = rbf_feature.fit_transform(X_scaled)
lr = LogisticRegression(C=1.0, max_iter=1000)
lr.fit(X_rbf, y)

# Построение сетки и контуров
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
X_grid = np.c_[xx.ravel(), yy.ravel()]
X_grid_scaled = scaler.transform(X_grid)

# Предсказания SVM
Z_svm = svm.predict(X_grid_scaled).reshape(xx.shape)
# Предсказания KLR (вероятность класса 1)
X_grid_rbf = rbf_feature.transform(X_grid_scaled)
Z_lr = lr.predict_proba(X_grid_rbf)[:, 1].reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.contourf(xx, yy, Z_svm, alpha=0.3, cmap='bwr')
ax1.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, edgecolors='k', cmap='bwr')
ax1.scatter(X_scaled[svm.support_, 0], X_scaled[svm.support_, 1],
            s=100, facecolors='none', edgecolors='k', linewidths=1.5, label='Опорные векторы')
ax1.set_title('SVM (RBF)')
ax1.legend()

ax2.contourf(xx, yy, Z_lr, levels=20, cmap='bwr', alpha=0.7)
ax2.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, edgecolors='k', cmap='bwr')
ax2.set_title('Ядерная логистическая регрессия (приближение)')

plt.tight_layout()
plt.show()



#### Вопросы для самопроверки

**Для начинающих:**

1. Почему ядерная логистическая регрессия считается ядровым методом? Какая теорема позволяет записать решение в виде $f(\mathbf{x}) = \sum_i \alpha_i K(\mathbf{x}, \mathbf{x}_i)$?
2. Как выглядит регуляризатор в задаче KLR и какую роль он играет? Запишите целевую функцию.
3. В чём отличие KLR от SVM с точки зрения получаемого решения (разреженность) и вида функции потерь?
4. Почему KLR выдаёт вероятности принадлежности классу, а SVM — нет? Как из KLR получить $p(y=+1 \mid \mathbf{x})$?
5. Чем функция потерь в KLR принципиально отличается от квадратичной потери в KRR?

**Для профессионалов:**

1. Выведите градиент и гессиан целевой функции KLR по $\boldsymbol{\alpha}$. Покажите, что гессиан представим в виде $\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}$, и объясните, почему он положительно определён.
2. Объясните, почему матрица $\mathbf{K}$ может быть плохо обусловлена, и как регуляризация $\lambda$ и метод IRLS помогают смягчить эту проблему. Какие ещё приёмы используются?
3. Сравните вычислительную сложность обучения KLR и KRR. Почему KLR требует итераций, а KRR даёт решение за один шаг?
4. Покажите связь KLR с гауссовскими процессами классификации: в каком смысле KLR является приближением к байесовскому выводу с логистической функцией связи?
5. Приведите пример задачи, где KLR демонстрирует значительно лучшую калибровку вероятностей, чем SVM с Platt scaling. Объясните, чем обусловлено преимущество.

### 5. Реализация ядерной логистической регрессии на Python

Теоретические основы ядерной логистической регрессии естественно воплощаются в компактную программную реализацию. Мы создадим класс `KernelLogisticRegression`, реализующий обучение через итеративно перевзвешенный метод наименьших квадратов (IRLS), протестируем его на классической проблеме XOR, сравним с полиномиальной логистической регрессией на реальных данных и исследуем влияние гиперпараметров.

Запустите следующую ячейку — она содержит полную реализацию класса.

In [ ]:
# @title Класс KernelLogisticRegression (IRLS, устойчивая версия)
import numpy as np
from scipy.spatial.distance import cdist
from scipy.linalg import solve

class KernelLogisticRegression:
    def __init__(self, kernel='rbf', gamma=1.0, degree=3, coef0=0.0, lambda_=1.0,
                 max_iter=50, tol=1e-6):
        self.kernel = kernel
        self.gamma = gamma
        self.degree = degree
        self.coef0 = coef0
        self.lambda_ = lambda_
        self.max_iter = max_iter
        self.tol = tol
        self.alpha_ = None
        self.X_train_ = None

    def _compute_kernel(self, X, Y=None):
        if Y is None:
            Y = X
        if self.kernel == 'linear':
            return X @ Y.T
        elif self.kernel == 'poly':
            return (self.gamma * (X @ Y.T) + self.coef0) ** self.degree
        elif self.kernel == 'rbf':
            sq_dists = cdist(X, Y, 'sqeuclidean')
            return np.exp(-self.gamma * sq_dists)
        elif self.kernel == 'sigmoid':
            return np.tanh(self.gamma * (X @ Y.T) + self.coef0)
        else:
            raise ValueError(f"Unknown kernel: {self.kernel}")

    def fit(self, X, y):
        n = X.shape[0]
        self.X_train_ = X
        K = self._compute_kernel(X)

        alpha = np.zeros(n)
        for it in range(self.max_iter):
            f = K @ alpha
            # y in {-1, +1}, p = sigma(y_i * f_i)
            p = 1.0 / (1.0 + np.exp(-y * f))
            w = p * (1.0 - p)
            w = np.clip(w, 1e-12, None)          # численная устойчивость
            z = f - (p - (y == 1)) / w            # рабочий отклик

            W = np.diag(w)
            # Система: (K + lambda * W^{-1}) beta = z, где beta = K @ alpha
            Winv = np.diag(1.0 / w)
            A = K + self.lambda_ * Winv
            b = z
            try:
                beta = solve(A, b, assume_a='pos')
            except np.linalg.LinAlgError:
                # дополнительная регуляризация при плохой обусловленности
                beta = solve(A + 1e-6 * np.eye(n), b, assume_a='pos')

            # Восстанавливаем alpha из beta = K @ alpha
            alpha_new = solve(K + 1e-6 * np.eye(n), beta, assume_a='pos')

            if np.linalg.norm(alpha_new - alpha) < self.tol:
                alpha = alpha_new
                break
            alpha = alpha_new

        self.alpha_ = alpha
        return self

    def predict_proba(self, X):
        K_new = self._compute_kernel(X, self.X_train_)
        f = K_new @ self.alpha_
        return 1.0 / (1.0 + np.exp(-f))

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.where(proba >= 0.5, 1, -1)

#### Тестирование на проблеме XOR

Классическая проблема XOR: четыре точки с координатами $(\pm1, \pm1)$, класс $+1$ если знаки координат совпадают. Линейная логистическая регрессия неспособна разделить такие данные. Ядерная логистическая регрессия с RBF-ядром строит нелинейную границу и справляется с задачей. Запустите ячейку ниже.

In [ ]:
# @title XOR: визуализация границы решений KLR
import matplotlib.pyplot as plt

X_xor = np.array([[1,1], [1,-1], [-1,1], [-1,-1]])
y_xor = np.array([1, -1, -1, 1])

model_xor = KernelLogisticRegression(kernel='rbf', gamma=1.0, lambda_=0.1)
model_xor.fit(X_xor, y_xor)

xx, yy = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
X_grid = np.c_[xx.ravel(), yy.ravel()]
Z = model_xor.predict_proba(X_grid).reshape(xx.shape)

plt.figure(figsize=(7,6))
plt.contourf(xx, yy, Z, levels=20, cmap='bwr', alpha=0.7)
plt.scatter(X_xor[:,0], X_xor[:,1], c=y_xor, cmap='bwr', edgecolors='k', s=100)
plt.title('Ядерная логистическая регрессия (RBF) на XOR')
plt.show()

#### Сравнение с полиномиальной логистической регрессией на breast cancer

Сравним качество вероятностных прогнозов ядерной логистической регрессии и обычной логистической регрессии с полиномиальными признаками. Оценим ROC-AUC и построим калибровочные кривые. Запустите ячейку.

In [ ]:
# @title Сравнение KLR и полиномиальной LR на breast cancer
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.calibration import calibration_curve

data = load_breast_cancer()
X, y = data.data, data.target
y = 2 * y - 1  # в -1, +1

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Ядерная LR
klr = KernelLogisticRegression(kernel='rbf', gamma=0.1, lambda_=1.0, max_iter=100)
klr.fit(X_train, y_train)
p_klr = klr.predict_proba(X_test)
auc_klr = roc_auc_score(y_test, p_klr)

# Полиномиальная LR (степень 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_poly, y_train)
p_lr = lr.predict_proba(X_test_poly)[:, 1]
auc_lr = roc_auc_score(y_test, p_lr)

print(f"KLR AUC: {auc_klr:.4f}")
print(f"Poly LR AUC: {auc_lr:.4f}")

plt.figure(figsize=(8,6))
for probs, label in [(p_klr, 'KLR'), (p_lr, 'Poly LR')]:
    frac_pos, mean_pred = calibration_curve(y_test == 1, probs, n_bins=10)
    plt.plot(mean_pred, frac_pos, 's-', label=label)
plt.plot([0,1],[0,1],'k:', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.legend()
plt.grid(True, alpha=0.3)
plt.title('Calibration curves')
plt.show()

#### Влияние $\lambda$ и $\gamma$ на качество

Интерактивный слайдер позволяет исследовать, как меняется решающая граница при изменении $\lambda$ (регуляризация) и $\gamma$ (ширина ядра) на данных типа «луны». Запустите ячейку и подвигайте слайдеры.

In [ ]:
# @title Влияние λ и γ на KLR (интерактивно)
from sklearn.datasets import make_moons
from ipywidgets import interact, FloatLogSlider

X_moons, y_moons = make_moons(n_samples=100, noise=0.25, random_state=42)
y_moons = 2 * y_moons - 1

def plot_klr(lambda_=1.0, gamma=1.0):
    model = KernelLogisticRegression(kernel='rbf', gamma=gamma, lambda_=lambda_)
    model.fit(X_moons, y_moons)
    xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.figure(figsize=(7,6))
    plt.contourf(xx, yy, Z, levels=20, cmap='bwr', alpha=0.7)
    plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap='bwr', edgecolors='k')
    plt.title(f'KLR (RBF) λ={lambda_:.2f}, γ={gamma:.2f}')
    plt.show()

interact(plot_klr,
         lambda_=FloatLogSlider(value=1.0, base=10, min=-3, max=2, step=0.1, description='λ'),
         gamma=FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.1, description='γ'));

#### Ограничения и приближённые методы

Как и другие ядровые методы, KLR страдает от кубической сложности $O(n^3)$ на итерацию и квадратичной памяти для матрицы $\mathbf{K}$. При $n \lesssim 10^4$ это приемлемо, но для больших выборок применяют аппроксимации: Nyström, случайные признаки (Random Fourier Features) или итерационные решатели (сопряжённые градиенты). Кроме того, число итераций IRLS обычно невелико (5–20), что делает метод конкурентоспособным.




#### Вопросы для самопроверки

**Для начинающих:**
1. Как метод IRLS применяется к ядерной логистической регрессии? Какие шаги он включает?
2. Зачем в IRLS вводятся веса $w_i$ и как они вычисляются?
3. Как изменится формула обновления для многоклассового случая (softmax-функция)?
4. Почему алгоритм IRLS может не сходиться? Какие численные проблемы могут возникнуть?

**Для профессионалов:**
1. Выведите матричное уравнение для обновления $\boldsymbol{\alpha}$ в IRLS для KLR и покажите, как оно сводится к $(\mathbf{W} \mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{W} \mathbf{z}$ при невырожденной $\mathbf{K}$.
2. Реализуйте альтернативный оптимизатор на основе LBFGS, не требующий решения линейной системы. Сравните число итераций и устойчивость.
3. Добавьте в класс функциональность ранней остановки по валидационной выборке: как отслеживать log loss на отложенных данных и прекращать обучение при его возрастании?
4. Объясните, как адаптировать KLR для мультиклассовой классификации (multinomial). Какие изменения вносятся в функцию потерь и в метод IRLS?
5. Сравните скорость сходимости IRLS и обычного градиентного спуска при обучении KLR. В каких ситуациях градиентный спуск может оказаться предпочтительнее?

### 6. Сравнение Kernel PCA, KLR, KRR и SVM на реальных данных

В предыдущих разделах мы детально изучили четыре ядровых метода: Kernel PCA (снижение размерности и визуализация), Kernel Ridge Regression (регрессия), Kernel Logistic Regression (вероятностная классификация) и SVM (классификация с максимизацией зазора). Теперь пришло время столкнуть их на реальной задаче, чтобы увидеть сильные и слабые стороны каждого и выработать практические рекомендации. В качестве тестового полигона мы используем подвыборку рукописных цифр MNIST — игрушечную, но достаточно богатую для демонстрации нелинейных эффектов.


#### 6.1. Визуализация MNIST с помощью Kernel PCA

MNIST содержит 70 000 изображений цифр $28 \times 28$ (784 пикселя). Для наглядности возьмём подмножество из 1000 случайных точек, охватывающее все 10 классов, и применим два метода снижения размерности до двух компонент.

- **Линейный PCA** просто ищет направления максимальной дисперсии в исходном 784-мерном пространстве. Двумерная проекция сохраняет глобальную структуру, но классы заметно перемешаны: цифры разных типов образуют перекрывающиеся облака, особенно схожие по начертанию (например, 4 и 9). Объяснённая дисперсия первых двух компонент невелика (около 15–20%), и визуализация не выявляет чёткой кластерной структуры.

- **Kernel PCA с RBF-ядром** ($\gamma \approx 10^{-3} \dots 10^{-2}$, подобранным по разбросу данных) даёт существенно более выразительное вложение. Благодаря нелинейному отображению в RKHS, классы становятся компактнее и разделяются заметно лучше. Цифры «0», «1», «7» часто образуют изолированные группы; сложные цифры (3, 5, 8) демонстрируют более чёткую внутреннюю структуру. Проекция на первые две ядерные главные компоненты визуально приближается к тому, что дают нелинейные методы типа t-SNE, но, в отличие от них, Kernel PCA имеет явную формулу преобразования и может применяться к новым точкам без переобучения.

Вывод: Kernel PCA с правильно настроенным ядром способен улавливать нелинейные многообразия, что делает его мощным инструментом визуализации высокоразмерных данных.




#### Визуализация Kernel PCA на MNIST

Запустите ячейку ниже, чтобы сравнить линейный PCA и Kernel PCA с RBF-ядром на подвыборке MNIST. Обратите внимание, как Kernel PCA лучше разделяет классы.

In [ ]:
# @title Kernel PCA vs линейный PCA на MNIST
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler

# Загружаем подвыборку MNIST (1000 точек)
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
idx = np.random.RandomState(42).choice(len(X), 1000, replace=False)
X, y = X[idx], y[idx].astype(int)
X_scaled = StandardScaler().fit_transform(X)

# Линейный PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Kernel PCA с RBF
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.01)
X_kpca = kpca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, X_proj, title in zip(axes, [X_pca, X_kpca], ['Линейный PCA', 'Kernel PCA (RBF)']):
    scatter = ax.scatter(X_proj[:, 0], X_proj[:, 1], c=y, cmap='tab10', alpha=0.6, s=10)
    ax.set_title(title)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
plt.colorbar(scatter, ax=axes, label='Digit')
plt.tight_layout()
plt.show()


#### 6.2. Классификация: KLR против SVM

Для бинарной классификации выделим цифры 0 и 1 (наиболее легко разделимые). Обучим на 500 точках и протестируем на 200. Применим KLR и SVM, оба с RBF-ядром.

- **SVM** быстро обучается (особенно с библиотекой LIBSVM/SVC) и даёт разреженное решение: из 500 обучающих точек опорными становятся лишь несколько десятков. Точность классификации на тесте близка к 100%, граница решения гладкая, но сами значения решающей функции не откалиброваны: предсказание лежит в диапазоне, не соответствующем вероятности.

- **KLR** оптимизирует логарифмическую потерю, что требует решения плотной задачи — в пределе все $\alpha_i$ отличны от нуля. Обучение IRLS сходится за 10–15 итераций, точность также около 100%, но предсказанные вероятности $p(y=+1 \mid \mathbf{x})$ хорошо откалиброваны: в областях, где классы смешаны (например, вблизи границы), вероятности плавно меняются от 0 до 1, в то время как SVM выдаёт значения, не имеющие вероятностного смысла. Время обучения KLR при $n=500$ пренебрежимо мало, но уже при $n=5000$ кубическая сложность даёт о себе знать: SVM обучается быстрее из-за разреженности и эффективных солверов, KLR требует хранения и обращения полной матрицы $\mathbf{K}$.

Таким образом, при малых и средних выборках KLR предпочтителен, когда важна вероятностная интерпретация. SVM выигрывает при больших $n$ и когда достаточно лишь метки класса.



#### Сравнение точности KLR и SVM на цифрах 0 и 1

Обучим оба классификатора на 500 точках и сравним точность и время обучения. KLR выдаёт вероятности, SVM — отступы.

In [ ]:
# @title Сравнение KLR и SVM на MNIST (0 vs 1) – исправлено
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import time

# Бинарная задача: цифры 0 и 1
mask = (y == 0) | (y == 1)
X_bin, y_bin = X_scaled[mask], y[mask]
y_bin = 2 * y_bin - 1  # в -1, +1

# Проверка количества объектов
print(f"Всего объектов (0 и 1): {len(X_bin)}")

# Разделение на обучение и тест (70/30)
X_tr, X_te, y_tr, y_te = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

# KLR
klr = KernelLogisticRegression(kernel='rbf', gamma=0.05, lambda_=1.0, max_iter=50)
start = time.time()
klr.fit(X_tr, y_tr)
klr_time = time.time() - start
klr_acc = accuracy_score(y_te, klr.predict(X_te))

# SVM
svm = SVC(kernel='rbf', gamma=0.05, C=1.0)
start = time.time()
svm.fit(X_tr, y_tr)
svm_time = time.time() - start
svm_acc = accuracy_score(y_te, svm.predict(X_te))

print(f"KLR: accuracy={klr_acc:.3f}, time={klr_time:.3f}s")
print(f"SVM: accuracy={svm_acc:.3f}, time={svm_time:.3f}s")



#### 6.3. Конвейер «снижение размерности + классификация»

Часто на практике применяют двухэтапный подход: сначала Kernel PCA сокращает размерность, затем классификатор (линейный или ядерный) обучается на полученных проекциях. Протестируем это на полном наборе из 10 классов MNIST (уже не бинарном). Для многоклассовой классификации будем использовать One-vs-Rest SVM и мультиномиальную логистическую регрессию с ядром (приближённую через случайные признаки).

1. **Без снижения размерности:** SVM с RBF-ядром на 784-мерных данных даёт высокое качество ($>98\%$), но обучение требует тщательной настройки $\gamma$ и $C$.
2. **Kernel PCA (50 компонент) + линейная логистическая регрессия:** проект на первые 50 ядерных главных компонент и затем применяем мультиклассовую логистическую регрессию. Качество может оказаться сравнимым с прямым SVM, но обучение значительно быстрее (особенно предсказание, так как компоненты предвычислены). Более того, 50-мерное представление можно использовать повторно для разных классификаторов.
3. **Прямой KLR с RBF** (в бинарном случае) и KLR на проекциях: прямое KLR даёт максимальную гибкость, но ценой плотного представления; KLR на проекциях проигрывает в точности лишь доли процента, но выигрывает в скорости.

Это демонстрирует, что Kernel PCA может служить не только для визуализации, но и как этап feature extraction, ускоряющий последующие алгоритмы за счёт работы в низкоразмерном пространстве, сохраняющем нелинейную структуру.




#### Эксперимент: Kernel PCA + линейный классификатор vs SVM

Проверим, как снижение размерности с помощью Kernel PCA влияет на точность и скорость. На полном наборе из 10 классов MNIST сравним прямой SVM с RBF‑ядром и конвейер: Kernel PCA (50 компонент) + мультиклассовая логистическая регрессия.

In [ ]:
# @title Конвейер Kernel PCA + LogisticRegression vs SVM на MNIST (10 классов)
from sklearn.decomposition import PCA, KernelPCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

# Загружаем подвыборку из 2000 точек для ускорения
X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
idx = np.random.RandomState(42).choice(len(X), 2000, replace=False)
X, y = X[idx], y[idx].astype(int)
X_scaled = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# 1. Прямой SVM с RBF
svm = SVC(kernel='rbf', gamma=0.01, C=1.0, random_state=42)
start = time.time()
svm.fit(X_train, y_train)
svm_time = time.time() - start
svm_acc = accuracy_score(y_test, svm.predict(X_test))
print(f"SVM (прямой, RBF): accuracy={svm_acc:.3f}, time={svm_time:.2f}s")

# 2. Kernel PCA (50 компонент) + логистическая регрессия
kpca = KernelPCA(n_components=50, kernel='rbf', gamma=0.01)
X_train_kpca = kpca.fit_transform(X_train)
X_test_kpca = kpca.transform(X_test)

lr = LogisticRegression(max_iter=500)
start = time.time()
lr.fit(X_train_kpca, y_train)
lr_time = time.time() - start
lr_acc = accuracy_score(y_test, lr.predict(X_test_kpca))
print(f"Kernel PCA(50) + LogisticRegression: accuracy={lr_acc:.3f}, time={lr_time:.2f}s")


#### 6.4. Выбор гиперпараметров кросс-валидацией

Общая стратегия для всех ядровых методов: сеточный поиск по $\gamma$ (для RBF) и параметру регуляризации ($\lambda$ в KRR/KLR, $C$ в SVM) с использованием $k$-фолд кросс-валидации.

- **KRR, KLR:** минимизируем log loss или RMSE на отложенных фолдах. Для KLR удобно отслеживать сходимость и останавливаться при росте валидационной ошибки (ранняя остановка).
- **SVM:** как правило, оптимизируют accuracy или F1-score; $C$ и $\gamma$ ищут на логарифмической сетке.
- **Kernel PCA:** число компонент $d$ выбирают по накопленной объяснённой дисперсии в $\mathcal{H}$ (сумма $\lambda_k / \sum \lambda_k$) или по downstream-задаче (например, точности классификации на проекциях). $\gamma$ также подбирают кросс-валидацией, иногда визуальным анализом кластеризации.

При реализации удобно использовать `GridSearchCV` из sklearn, оборачивая наши собственные классы KLR, KRR в интерфейс, совместимый с sklearn.





#### 6.5. Выводы: когда какой метод предпочтительнее

Сведём полученные наблюдения в практические рекомендации.

| Метод          | Основное применение                         | Вероятности | Разреженность | Масштабируемость      |
|----------------|---------------------------------------------|-------------|---------------|-----------------------|
| **KRR**        | Нелинейная регрессия, гладкие функции        | Нет         | Плотное       | $O(n^3)$, аппроксимации |
| **KLR**        | Классификация с калиброванными вероятностями | Да          | Плотное       | $O(n^3)$, IRLS/Newton  |
| **SVM**        | Классификация с максимизацией зазора         | Нет (Platt) | Разреженное   | $O(n^2)$–$O(n^3)$      |
| **Kernel PCA** | Визуализация, денузинг, feature extraction    | —           | Плотное       | $O(n^3)$, Nyström      |

**Когда выбирать каждый из них:**

- *Kernel PCA*: если нужно исследовать нелинейную структуру данных, визуализировать многомерные облака, очистить данные от шума или подготовить компактное нелинейное представление для других методов.
- *KRR*: когда решается задача регрессии, отклик непрерывен, а связь с признаками предположительно нелинейна; когда важна аналитическая форма решения и его байесовская интерпретация (гауссовские процессы).
- *KLR*: если нужны именно вероятности принадлежности классам (оценка рисков, кредитный скоринг), и размер выборки не превышает нескольких десятков тысяч. Даёт хорошо откалиброванные прогнозы.
- *SVM*: когда первостепенна точность разделения классов, выборка велика, и вероятностный выход не обязателен. Разреженность SVM ускоряет предсказание и снижает потребление памяти.

В реальных проектах часто комбинируют методы: например, Kernel PCA + KLR/SVM, или используют несколько ядер с Multiple Kernel Learning. Универсального «лучшего» метода нет — выбор диктуется природой данных, требуемой интерпретируемостью и вычислительными ресурсами.





#### Вопросы для самопроверки

**Для начинающих:**

1. Какую роль играет Kernel PCA в конвейере анализа данных? Приведите пример, когда он полезен перед классификацией.
2. Почему снижение размерности с помощью Kernel PCA может улучшить качество классификации? В каких случаях оно не помогает?
3. Как выбрать число главных компонент в Kernel PCA? Какие критерии вы знаете?
4. Сравните время обучения KLR и SVM при $n = 5000$. Почему SVM обычно быстрее?
5. Что такое «предвычисленное ядро» и как оно помогает ускорить подбор гиперпараметров?

**Для профессионалов:**

1. Проведите анализ чувствительности Kernel PCA к выбросам: как наличие нескольких аномальных точек влияет на центрирование и главные компоненты? Предложите робастную модификацию.
2. Сравните Kernel PCA с автокодировщиками (нейронными сетями) для нелинейного снижения размерности. В каких условиях каждый из них предпочтительнее?
3. Предложите метод ускорения обучения KLR через аппроксимацию ядерной матрицы (например, Nyström) и опишите, как изменится уравнение IRLS.
4. Объясните, как ядерная логистическая регрессия справляется с несбалансированными классами. Достаточно ли ввести веса примеров в функцию потерь, и как это повлияет на IRLS?
5. Реализуйте KLR с использованием случайных признаков (Random Fourier Features) для RBF-ядра. Выпишите целевую функцию и градиентный шаг для стохастического спуска.